# 02: Similarity Search

Build a semantic search system using embeddings and similarity metrics.

## Prerequisites

- Python 3.10+
- `openai`, `numpy`, `scikit-learn` installed
- A OpenAI API key

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
import openai
import numpy as np

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=API_KEY)

EMBEDDING_MODEL = "models/text-embedding-3-small"
print("Ready.")

## 2. Cosine Similarity from Scratch

Implement cosine similarity using only NumPy.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def dot_product_similarity(a, b):
    """Compute dot product similarity."""
    return np.dot(a, b)


def euclidean_similarity(a, b):
    """Compute Euclidean distance-based similarity (1 / (1 + distance))."""
    distance = np.linalg.norm(a - b)
    return 1 / (1 + distance)


# Test with sample vectors
a = np.array([1.0, 0.0, 0.0])
b = np.array([0.0, 1.0, 0.0])
c = np.array([0.7, 0.7, 0.0])

print("a vs b (orthogonal):")
print(f"  Cosine: {cosine_similarity(a, b):.4f}")
print(f"  Dot:    {dot_product_similarity(a, b):.4f}")
print(f"  Euclid: {euclidean_similarity(a, b):.4f}")

print("\na vs c (partially aligned):")
print(f"  Cosine: {cosine_similarity(a, c):.4f}")
print(f"  Dot:    {dot_product_similarity(a, c):.4f}")
print(f"  Euclid: {euclidean_similarity(a, c):.4f}")

## 3. Building a Document Index

Create a reusable search engine class.

In [ ]:
class SemanticSearchEngine:
    """A simple semantic search engine using Gemini embeddings."""
    
    def __init__(self, model=EMBEDDING_MODEL):
        self.model = model
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, documents):
        """Add documents and generate their embeddings."""
        self.documents.extend(documents)
        
        # Generate embeddings
        result = genai.embed_content(
            model=self.model,
            content=documents,
        )
        new_embeddings = np.array(result["embedding"])
        
        # Stack with existing embeddings
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
        print(f"Indexed {len(documents)} documents. Total: {len(self.documents)}")
    
    def search(self, query, top_k=5, metric="cosine"):
        """Search documents by semantic similarity."""
        # Embed query
        result = genai.embed_content(
            model=self.model,
            content=query,
            task_type="RETRIEVAL_QUERY",
        )
        query_embedding = np.array(result["embedding"])
        
        # Compute similarities
        if metric == "cosine":
            similarities = np.array([
                cosine_similarity(query_embedding, doc_emb)
                for doc_emb in self.embeddings
            ])
        elif metric == "dot":
            similarities = np.array([
                dot_product_similarity(query_embedding, doc_emb)
                for doc_emb in self.embeddings
            ])
        elif metric == "euclidean":
            similarities = np.array([
                euclidean_similarity(query_embedding, doc_emb)
                for doc_emb in self.embeddings
            ])
        else:
            raise ValueError(f"Unknown metric: {metric}")
        
        # Get top-k indices
        top_indices = similarities.argsort()[::-1][:top_k]
        
        return [
            {"document": self.documents[i], "score": float(similarities[i])}
            for i in top_indices
        ]
    
    def __len__(self):
        return len(self.documents)

## 4. Indexing a Knowledge Base

In [ ]:
# Sample knowledge base (tech support scenarios)
knowledge_base = [
    "Python ImportError occurs when a module cannot be found. Install it with pip install <package>.",
    "To fix a NameError, check that the variable is defined before use.",
    "Memory errors happen when your program uses too much RAM. Try optimizing data structures.",
    "To resolve a TypeError, ensure variables are the correct type before operations.",
    "For connection timeout errors, check your network settings and server availability.",
    "SyntaxError means there's a grammar mistake in your Python code. Check indentation.",
    "To fix permission denied errors, check file permissions or run with appropriate privileges.",
    "IndexError occurs when accessing a list element beyond its bounds. Check array length.",
    "For CUDA out of memory errors, reduce batch size or use GPU with more VRAM.",
    "To resolve circular import errors, restructure your imports or use lazy loading.",
    "KeyError happens when accessing a dictionary key that doesn't exist. Use .get() method.",
    "For API rate limiting, implement exponential backoff and retry logic.",
]

# Create search engine and index documents
engine = SemanticSearchEngine()
engine.add_documents(knowledge_base)
print(f"\nIndexed {len(engine)} documents")

## 5. Performing Searches

In [ ]:
# Test searches with natural language queries
queries = [
    "my code can't find a package",
    "the program runs out of memory",
    "I can't connect to the server",
    "the GPU is full",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    results = engine.search(query, top_k=3)
    for i, result in enumerate(results, 1):
        print(f"  {i}. [{result['score']:.4f}] {result['document'][:80]}...")

### Exercise: Search Analysis

Try these queries and analyze the results:
1. "variable is not defined" — what comes up?
2. "network problems" — does it find connection errors?
3. "I forgot a comma in my code" — does it find syntax errors?
4. Create your own query and test it.

## 6. Comparing Similarity Metrics

In [ ]:
query = "Python package not found"

print(f"Query: '{query}'\n")
print(f"{'Rank':<5} {'Cosine':<10} {'Dot':<10} {'Euclidean':<10} Document")
print("-" * 80)

# Get results from each metric
cosine_results = engine.search(query, top_k=len(engine), metric="cosine")
dot_results = engine.search(query, top_k=len(engine), metric="dot")
euclidean_results = engine.search(query, top_k=len(engine), metric="euclidean")

# Show top 5 from each
for i in range(5):
    cos_doc = cosine_results[i]["document"][:35]
    cos_score = cosine_results[i]["score"]
    dot_score = dot_results[i]["score"]
    euc_score = euclidean_results[i]["score"]
    print(f"{i+1:<5} {cos_score:<10.4f} {dot_score:<10.4f} {euc_score:<10.4f} {cos_doc}...")

### Exercise: Metric Comparison

1. Run the same query with all three metrics. Do they return the same top-3 results?
2. Try a query where you expect the results to differ. Why might they differ?
3. When would you choose dot product over cosine similarity?

## 7. Similarity Matrix

Visualize how all documents relate to each other.

In [ ]:
import matplotlib.pyplot as plt

# Compute pairwise similarity matrix
n = len(engine)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity_matrix[i, j] = cosine_similarity(
            engine.embeddings[i], engine.embeddings[j]
        )

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(similarity_matrix, cmap="YlOrRd", vmin=0.3, vmax=1.0)
ax.set_title("Document Similarity Matrix")
ax.set_xlabel("Document Index")
ax.set_ylabel("Document Index")
plt.colorbar(im, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 8. Building a Recommendation System

Recommend similar articles based on embedding similarity.

In [ ]:
# Sample articles
articles = [
    "Getting started with Python for data science",
    "Introduction to machine learning algorithms",
    "Building neural networks with PyTorch",
    "Web scraping with BeautifulSoup and Python",
    "Natural language processing fundamentals",
    "Data visualization with Matplotlib and Seaborn",
    "Deploying ML models to production",
    "Understanding transformers in deep learning",
    "Feature engineering techniques for ML",
    "Python best practices for clean code",
]

# Index articles
recommendation_engine = SemanticSearchEngine()
recommendation_engine.add_documents(articles)

In [ ]:
def recommend(article_title, engine, top_k=3):
    """Recommend similar articles."""
    results = engine.search(article_title, top_k=top_k + 1)  # +1 to exclude itself
    
    print(f"\nBecause you liked: '{article_title}'")
    print("Recommended:")
    for i, result in enumerate(results[1:], 1):  # Skip first (itself)
        print(f"  {i}. [{result['score']:.3f}] {result['document']}")


# Test recommendations
recommend("Introduction to machine learning algorithms", recommendation_engine)
recommend("Web scraping with BeautifulSoup and Python", recommendation_engine)
recommend("Understanding transformers in deep learning", recommendation_engine)

## 9. Performance Considerations

In [ ]:
import time

# Benchmark: time to index documents
test_sizes = [10, 50, 100]

for size in test_sizes:
    test_docs = [f"Document number {i} about topic {i % 10}" for i in range(size)]
    
    start = time.time()
    result = genai.embed_content(model=EMBEDDING_MODEL, content=test_docs)
    elapsed = time.time() - start
    
    print(f"{size} documents: {elapsed:.2f}s ({elapsed/size*1000:.1f}ms per doc)")

### Optimization Tips

1. **Batch API calls**: Embed multiple texts at once (up to 100 per call)
2. **Cache embeddings**: Store embeddings in a file/database, don't recompute
3. **Use vector databases**: For large-scale search, use Pinecone, Weaviate, or Qdrant
4. **Reduce dimensions**: Use 256 or 512 dimensions if full 768 isn't needed
5. **Approximate nearest neighbors**: Use FAISS for faster search at scale

## Exercises

1. **Domain-specific search**: Pick a domain (e.g., healthcare, finance). Create 15+ documents and test search quality. What queries work well? What fails?

2. **Metric analysis**: For 10 different queries, compare cosine vs dot product vs Euclidean. Create a table showing agreement rate in top-3 results.

3. **Hybrid search**: Combine keyword matching (BM25-style) with semantic search. Do the results improve?

4. **Scale test**: Generate 200+ documents. Measure search latency. At what point do you need a vector database?

5. **Save and load**: Modify the `SemanticSearchEngine` to save/load embeddings to/from a JSON file.

## Summary

- Cosine similarity is the standard metric for embedding comparison
- Build a search engine by indexing document embeddings and comparing query embeddings
- Different similarity metrics can produce different rankings
- Semantic search finds relevant results even without exact keyword matches
- For production, consider vector databases and caching strategies

→ Next: [Embedding Comparison](03-embedding-comparison.ipynb)